# NYC Taxi Trip Duration — first model

Predict the length of a taxi ride in seconds. This is **regression**, not a future time-series forecast: train and test trips cover the same months. We use pickup information and destination coordinates, but never the actual drop-off time.


## Helper functions

These two reusable functions create model inputs and calculate Kaggle's score. Defining them here does not train a model yet.


In [ ]:
def build_features(df):
    pickup = pd.to_datetime(df['pickup_datetime'])
    lat1 = np.radians(df['pickup_latitude'].astype(float))
    lat2 = np.radians(df['dropoff_latitude'].astype(float))
    dlat = lat2 - lat1
    dlon = np.radians(df['dropoff_longitude'].astype(float) - df['pickup_longitude'].astype(float))
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2

    features = pd.DataFrame(index=df.index)
    features['distance_km'] = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    features['pickup_hour'] = pickup.dt.hour
    features['pickup_weekday'] = pickup.dt.dayofweek
    features['pickup_month'] = pickup.dt.month
    features['pickup_latitude'] = df['pickup_latitude']
    features['pickup_longitude'] = df['pickup_longitude']
    features['dropoff_latitude'] = df['dropoff_latitude']
    features['dropoff_longitude'] = df['dropoff_longitude']
    features['passenger_count'] = df['passenger_count']
    features['vendor_id'] = df['vendor_id']
    features['store_and_fwd'] = (df['store_and_fwd_flag'] == 'Y').astype(int)
    return features


In [ ]:
def rmsle(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.clip(np.asarray(predicted), 0, None)
    return np.sqrt(np.mean((np.log1p(actual) - np.log1p(predicted)) ** 2))


## 1. Load and inspect

Each row is one trip. `trip_duration` is the target. `dropoff_datetime` is available only after the trip and would reveal the answer, so it is never passed to the model.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor

data_dir = Path('/kaggle/input/nyc-taxi-trip-duration')
if not data_dir.exists():
    data_dir = Path('data')

def data_file(stem):
    csv_file = data_dir / f'{stem}.csv'
    return csv_file if csv_file.exists() else data_dir / f'{stem}.zip'

train = pd.read_csv(data_file('train'), parse_dates=['pickup_datetime'])
test = pd.read_csv(data_file('test'), parse_dates=['pickup_datetime'])
print('Train:', train.shape, 'Test:', test.shape)
print('Missing train:', train.isna().sum().to_dict())
print('Train dates:', train.pickup_datetime.min(), 'to', train.pickup_datetime.max())
print('Test dates:', test.pickup_datetime.min(), 'to', test.pickup_datetime.max())
print('Trip duration seconds:', train.trip_duration.quantile([0, .5, .99, 1]).to_dict())


## 2. Build features and validation

We use straight-line distance, departure hour/day/month, location, passenger count, and vendor. The competition's train and test cover the same six months, so a random holdout is a reasonable first approximation of its hidden test. This is unlike Store Sales, where we predicted future dates and had to split past → future. The destination coordinates are competition-provided; a real app could use them only after the rider supplies a destination.


In [ ]:
X = build_features(train)
X_test = build_features(test)
y = train['trip_duration'].to_numpy()
assert X.columns.equals(X_test.columns)
assert 'dropoff_datetime' not in X and 'trip_duration' not in X
train_idx, valid_idx = train_test_split(np.arange(len(train)), test_size=0.20, random_state=42)
print('Features:', X.columns.tolist())
print('Train rows:', len(train_idx), 'Validation rows:', len(valid_idx))


## 3. Baseline → model

The baseline predicts one typical duration for every trip. The model gets a limited training sample so this first pass runs quickly; it learns patterns from distance, place, and pickup time. Both are scored on the same untouched validation rows. Lower RMSLE is better.


In [ ]:
baseline_seconds = np.expm1(np.log1p(y[train_idx]).mean())
baseline_score = rmsle(y[valid_idx], np.full(len(valid_idx), baseline_seconds))

rng = np.random.default_rng(42)
sample_idx = rng.choice(train_idx, size=min(300_000, len(train_idx)), replace=False)
model = HistGradientBoostingRegressor(max_iter=120, max_leaf_nodes=31, learning_rate=0.08, random_state=42)
model.fit(X.iloc[sample_idx], np.log1p(y[sample_idx]))
train_prediction = np.expm1(model.predict(X.iloc[sample_idx])).clip(min=0)
train_score = rmsle(y[sample_idx], train_prediction)
valid_prediction = np.expm1(model.predict(X.iloc[valid_idx])).clip(min=0)
model_score = rmsle(y[valid_idx], valid_prediction)
print(f'Baseline RMSLE: {baseline_score:.5f}')
print(f'Model train RMSLE: {train_score:.5f}')
print(f'Model RMSLE:    {model_score:.5f}')


## 4. Predict test and create submission

In the first local run, the constant baseline scored 0.79661 RMSLE; gradient boosting scored 0.42202 on its training sample and 0.42871 on validation. That small train/validation gap is reassuring on this split, though it does not test future months. Only after this comparison do we retrain on a sample of all labeled trips. Check row count, IDs, missing values, and negative predictions before saving. A Kaggle late-submission score is practice feedback, not a result from the original 2017 competition.


In [ ]:
final_idx = rng.choice(np.arange(len(train)), size=min(500_000, len(train)), replace=False)
final_model = HistGradientBoostingRegressor(max_iter=120, max_leaf_nodes=31, learning_rate=0.08, random_state=42)
final_model.fit(X.iloc[final_idx], np.log1p(y[final_idx]))
test_prediction = np.expm1(final_model.predict(X_test)).clip(min=0)
submission = pd.DataFrame({'id': test['id'], 'trip_duration': test_prediction})
assert len(submission) == len(test)
assert submission['id'].equals(test['id'])
assert np.isfinite(submission['trip_duration']).all()
assert submission['trip_duration'].ge(0).all()
output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('output')
output_dir.mkdir(exist_ok=True)
submission.to_csv(output_dir / 'submission.csv', index=False)
print('Saved:', output_dir / 'submission.csv', 'Rows:', len(submission))
print(submission.head())
